In [2]:
import pandas as pd
import numpy as np
import ast
from nltk.stem import PorterStemmer

In [41]:
import pandas as pd
import ast
from nltk.stem import PorterStemmer

# 1. Load Data
movies = pd.read_csv('clean_movies.csv', on_bad_lines='skip', engine='python')

# 2. Conversion Functions
def convert(text):
    L = []
    try:
        for i in ast.literal_eval(text):
            L.append(i['name'])
    except:
        pass
    return L

def convert_cast(text):
    L = []
    counter = 0
    try:
        for i in ast.literal_eval(text):
            if counter != 3:
                L.append(i['name'])
                counter += 1
            else:
                break
    except:
        pass
    return L

def fetch_director(text):
    L = []
    try:
        for i in ast.literal_eval(text):
            if i['job'] == 'Director':
                L.append(i['name'])
                break
    except:
        pass
    return L

# 3. Apply Transformations
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: str(x).split())

# 4. Clean Spaces
for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(lambda x: [i.replace(' ', '') for i in x])

# 5. Create Tags and Final DF
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new_df = movies[['id', 'title', 'tags', 'vote_average', 'vote_count', 'popularity', 'release_date', 'original_language']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x)).str.lower()

# 6. Stemming
ps = PorterStemmer()
def stem(text):
    return ' '.join([ps.stem(word) for word in text.split()])

new_df['tags'] = new_df['tags'].apply(stem)

# 7. Save
new_df.to_csv('processed_movies.csv', index=False)
print('Data processing complete. Final file saved as processed_movies.csv.')
display(new_df.head())

Data processing complete. Final file saved as processed_movies.csv.


,id,title,tags,vote_average,vote_count,popularity,release_date,original_language
0,862,Toy Story,"led by woody, andy' toy live happili in hi roo...",7.7,5415.0,21.946943,1995-10-30,en
1,8844,Jumanji,when sibl judi and peter discov an enchant boa...,6.9,2413.0,17.015539,1995-12-15,en
2,15602,Grumpier Old Men,a famili wed reignit the ancient feud between ...,6.5,92.0,11.712900,1995-12-22,en
3,31357,Waiting to Exhale,"cheat on, mistreat and step on, the women are ...",6.1,34.0,3.859495,1995-12-22,en
4,11862,Father of the Bride Part II,just when georg bank ha recov from hi daughter...,5.7,173.0,8.387519,1995-02-10,en
